# RLSS2026 - PG Tutorial: PD Control and Black-Box Optimization (BBO)

Website: https://rlsummerschool.com/

Github repository: https://github.com/araffin/rlss26-pg-tutorial

Gymnasium documentation: https://gymnasium.farama.org/

## Introduction

In this notebook, you will implement a PD controller and optimize its gains automatically using black-box optimization to solve the line following problem.


In [ ]:
# for autoformatting
# !pip install jupyter-blaµck
%load_ext jupyter_black

## Install Dependencies

In [ ]:
import os

ON_COLAB = os.environ.get("COLAB_NOTEBOOK_ID") is not None

if ON_COLAB:
    !pip install git+https://github.com/araffin/rlss26-pg-tutorial --upgrade

In [ ]:
if ON_COLAB:
    !apt-get install ffmpeg  # For visualization

## Line Following Environment

For this tutorial, we will use the line following environment where a car try to follow a line to finish a racing track as fast as possible.



In [ ]:
from typing import Callable

import numpy as np

from pg_tutorial.envs.constant_speed import LineFollowerConstantSpeedEnv

# Available tracks: "oval`, "s_track", "rounded_l", "hairpin", "custom".
track_name = "s_track"

# Instantiate the environment
env = LineFollowerConstantSpeedEnv(track_name=track_name)

In [ ]:
# Box(1) means that there is one continuous action: steering
print("Action space:", env.action_space)

### Exercise (3 minutes): write a simple "bang-bang" controller

In [ ]:
def bang_bang_control(lateral_error: float, prev_lateral_error: float, steering_step: float = 0.3) -> float:
    if lateral_error >= 0:
        # We are on the right of the line, go left
        action = steering_step
    else:
        # Go right
        action = -steering_step
    # or
    # steering = np.sign(lateral_error) * steering_step
    return action

**HINT**: Take a look at the slide on Gym API or at the Gymnasium documentation: https://gymnasium.farama.org/

In [ ]:
def collect_data(env_id: str, n_steps: int = 50_000) -> OfflineData:
    """
    Collect transitions using a random agent (sample action randomly).

    :param env_id: The name of the environment.
    :param n_steps: Number of steps to perform in the env.
    :return: The collected transitions.
    """
    # Create the Gym env
    env = gym.make(env_id)

    assert isinstance(env.observation_space, spaces.Box)
    # Numpy arrays (buffers) to collect the data
    observations = np.zeros((n_steps, *env.observation_space.shape))
    next_observations = np.zeros((n_steps, *env.observation_space.shape))
    # Discrete actions
    actions = np.zeros((n_steps, 1))
    rewards = np.zeros((n_steps,))
    terminateds = np.zeros((n_steps,))

    # Variable to know if the episode is over (done = terminated or truncated)
    done = False

    ### YOUR CODE HERE
    # You need to collect transitions for `n_steps` using
    # a random agent (sample action uniformly).
    # Do not forget to reset the environment if the current episode is over
    # (done = terminated or truncated)
    #
    # TODO:
    # 1. Sample a random action
    # 2. Step in the env using this random action
    # 3. Retrieve the new transition data (observation, reward, ...)
    #  and update the numpy arrays (buffers)
    # 4. Repeat until you collected `n_steps` transitions

    # Start the first episode
    current_obs, _ = env.reset()
    
    for idx in range(n_steps):
        # Sample a random action

        # Step in the environment

        # Store the transition
        # Note: we only record true termination (timeouts/truncations are artificial terminations)

        
        # Update current observation

        # Check if the episode is over

        # Don't forget to reset the env at the end of an episode


    ### END OF YOUR CODE

    return OfflineData(
        observations,
        next_observations,
        actions,
        rewards,
        terminateds,
    )

Let's try the collect data method:

In [ ]:
# Check the length of the collected data
assert len(data.observations) == n_steps
assert len(data.actions) == n_steps
# Check that there are multiple episodes in the data
assert not np.all(data.terminateds)
assert np.any(data.terminateds)
# Check the shape of the collected data
if env_id == "CartPole-v1":
    assert data.observations.shape == (n_steps, 4)
    assert data.next_observations.shape == (n_steps, 4)
assert data.actions.shape == (n_steps, 1)
assert data.rewards.shape == (n_steps,)

### 2. Exercise (8 minutes): write the function to evaluate a model

A greedy policy $\pi(s)$ can be defined using the q-value:

$\pi(s) = argmax_{a \in A} Q(s, a)$.

It is the policy that takes the action with the highest q-value for a given state.

In [ ]:
%load_ext autoreload
%autoreload 2

In [57]:
import os

import gymnasium as gym
from gymnasium.wrappers import RecordVideo

IDX_LATERAL_ERROR: int = 0


def evaluate(
    controller: Callable,
    env: gym.Env,
    n_eval_episodes: int = 5,
    video_name: str | None = None,
    verbose: bool = True,
) -> tuple[float, float]:
    episode_returns, episode_reward = [], 0.0
    total_episodes = 0
    done = False

    # Setup video recorder
    if video_name is not None and env.render_mode == "rgb_array":
        # TODO: replace with abs path
        os.makedirs("../logs/videos/", exist_ok=True)

        # New gym recorder always wants to cut video into episodes,
        # set video length big enough but not to inf (will cut into episodes)
        env = RecordVideo(env, "../logs/videos", step_trigger=lambda _: False, video_length=100_000)
        env.start_recording(video_name)

    # some gym magic to retrieve the original env
    line_follower_env = env.unwrapped
    assert isinstance(line_follower_env, LineFollowerConstantSpeedEnv)

    obs, _ = env.reset()
    lateral_error = prev_lateral_error = float(obs[IDX_LATERAL_ERROR])
    best_lap_time = float("inf")
    lap_count = 0

    while total_episodes < n_eval_episodes:
        # retrieve prev and current lateral error
        lateral_error = obs[IDX_LATERAL_ERROR]
        action = controller(lateral_error, prev_lateral_error)

        obs, reward, terminated, truncated, info = env.step(action)
        prev_lateral_error = lateral_error

        episode_reward += float(reward)

        if info["lap_count"] > lap_count:
            lap_count = info["lap_count"]
            last_lap_time = info["last_lap_time"]
            best_lap_time = info["best_lap_time"]

        done = terminated or truncated
        if done:
            episode_returns.append(episode_reward)
            episode_reward = 0.0
            total_episodes += 1
            current_obs, _ = env.reset()

    if isinstance(env, RecordVideo):
        print(f"Saving video to ../logs/videos/{video_name}")
        env.close()
    if verbose:
        print(f"{best_lap_time=:.2f}s | Total reward = {np.mean(episode_returns):.2f} +/- {np.std(episode_returns):.2f}")
    return best_lap_time, np.mean(episode_returns)

In [58]:
# Evaluate the bang_bang_controller iteration
evaluate(bang_bang_control, env, n_eval_episodes=5)

best_lap_time=infs | Total reward = 33.01 +/- 0.62


(inf, np.float64(33.00808975390966))

### Record a video

In [ ]:
from pg_tutorial.notebook_utils import show_videos

# Available tracks: "oval`, "s_track", "rounded_l", "hairpin", "custom".
eval_env = LineFollowerConstantSpeedEnv(track_name="s_track", base_speed=0.9, render_mode="rgb_array")
video_name = "BANG_BANG"
n_eval_episodes = 2

evaluate(bang_bang_control, eval_env, n_eval_episodes, video_name=video_name)
show_videos("../logs/videos/", prefix=video_name)

In [ ]:
from dataclasses import dataclass


@dataclass
class PDController:
    kp: float = 0.0  # proportional gain
    kd: float = 0.0  # derivative gain
    dt: float = 1 / 60  # controller frequency

    def compute_action(self, lateral_error: float, prev_lateral_error: float) -> float:
        action = self.kp * lateral_error + self.kd * (lateral_error - prev_lateral_error) / self.dt
        # limit action
        action = np.clip(action, -1.0, 1.0)
        return action

    def __call__(self, lateral_error: float, prev_lateral_error: float) -> float:
        # a alias to be able to directly do controller(error, prev_error)
        return self.compute_action(lateral_error, prev_lateral_error)

In [ ]:
# Available tracks: "oval`, "s_track", "rounded_l", "hairpin", "custom".
eval_env = LineFollowerConstantSpeedEnv(
    track_name="oval",
    base_speed=0.5,
    render_mode="rgb_array",
    max_episode_steps=500,
)
video_name = "pd_controller"
n_eval_episodes = 1

# TODO(antonin): silence the video recorder warning
evaluate(PDController(kp=0.01, kd=0.01, dt=eval_env.dt), eval_env, n_eval_episodes, video_name=video_name)
# evaluate(PDController(kp=0.00420, kd=0.00758, dt=env.dt), eval_env, n_eval_episodes, video_name=video_name)
# TODO(antonin): print best lap time
show_videos("../logs/videos/", prefix=video_name)

In [59]:
# Optimize gains with BBO
# Available tracks: "oval`, "s_track", "rounded_l", "hairpin", "custom".
eval_env = LineFollowerConstantSpeedEnv(
    track_name="s_track",
    base_speed=0.5,
    max_episode_steps=500,
)
n_eval_episodes = 1

initial_kp = initial_kd = 0.0001
best_gains = np.array([initial_kp, initial_kd])
# best_gains = (initial_kp, initial_kd)
pop_size = 5
pop_std = 0.001
n_iterations = 10
best_lap_time = float("inf")

for iteration in range(1, n_iterations + 1):
    # Sample around the best_gains
    # Negative gains don't make sense
    # candidates = np.abs([best_gains + np.random.normal(0.0, pop_std, size=2) for _ in range(pop_size)])
    best_kp, best_kd = best_gains
    # for candidate in candidates:
    for _ in range(pop_size):
        kp = np.abs(best_kp + np.random.normal(0.0, pop_std))
        kd = np.abs(best_kd + np.random.normal(0.0, pop_std))
        lap_time, _ = evaluate(PDController(kp=kp, kd=kd, dt=eval_env.dt), eval_env, n_eval_episodes, verbose=False)
        if lap_time < best_lap_time:
            best_lap_time = lap_time
            best_gains = np.array([kp, kd])
    print(f"{iteration=} | {best_lap_time=:.2f}s | {best_gains=}")

iteration=1 | best_lap_time=12.37s | best_gains=array([0.00090061, 0.00063432])
iteration=2 | best_lap_time=11.43s | best_gains=array([0.00303519, 0.00276137])
iteration=3 | best_lap_time=11.23s | best_gains=array([0.00289435, 0.00375447])
iteration=4 | best_lap_time=11.17s | best_gains=array([0.00208967, 0.00476897])
iteration=5 | best_lap_time=11.03s | best_gains=array([0.0032577 , 0.00675844])
iteration=6 | best_lap_time=11.00s | best_gains=array([0.00428707, 0.0076995 ])
iteration=7 | best_lap_time=11.00s | best_gains=array([0.00428707, 0.0076995 ])
iteration=8 | best_lap_time=11.00s | best_gains=array([0.00428707, 0.0076995 ])
iteration=9 | best_lap_time=11.00s | best_gains=array([0.00428707, 0.0076995 ])
iteration=10 | best_lap_time=11.00s | best_gains=array([0.00428707, 0.0076995 ])


In [81]:
# Optimize gains with finite diff
# Available tracks: "oval`, "s_track", "rounded_l", "hairpin", "custom".
eval_env = LineFollowerConstantSpeedEnv(
    track_name="s_track",
    base_speed=0.5,
    max_episode_steps=500,
)
n_eval_episodes = 1

initial_kp = initial_kd = 0.0001
best_gains = np.array([initial_kp, initial_kd])
# best_gains = (initial_kp, initial_kd)
n_deltas = 1
epsilon = 0.0001
n_iterations = 20
best_lap_time = float("inf")
learning_rate = 0.001

for iteration in range(1, n_iterations + 1):
    for _ in range(n_deltas):
        direction = np.random.normal((0.0, 1.0), size=2)
        # normalize
        direction /= np.linalg.norm(direction)

        gains_plus = best_gains + epsilon * direction
        gains_minus = best_gains - epsilon * direction

        lap_time_plus, reward_plus = evaluate(
            PDController(kp=gains_plus[0], kd=gains_plus[1], dt=eval_env.dt), eval_env, n_eval_episodes, verbose=False
        )
        lap_time_minus, reward_minus = evaluate(
            PDController(kp=gains_minus[0], kd=gains_minus[1], dt=eval_env.dt), eval_env, n_eval_episodes, verbose=False
        )
        # TODO: use reward here instead?
        # grad = (1 / (n_deltas * epsilon)) * (reward_plus - reward_minus) * direction
        grad = (reward_plus - reward_minus) * direction

        print(grad, direction)
        # TODO: normalize grad (grad clip)
        best_gains = np.abs(best_gains - learning_rate * grad)

        if lap_time_plus < best_lap_time:
            best_lap_time = lap_time_plus
        if lap_time_minus < best_lap_time:
            best_lap_time = lap_time_minus
            # best_gains = np.array([kp, kd])
    print(f"{iteration=} | {best_lap_time=:.2f}s | {best_gains=}")

[-1.93186719  5.70542771] [-0.32071524  0.94717566]
iteration=1 | best_lap_time=infs | best_gains=array([0.00203187, 0.00560543])
[ 0.47347875 -0.22021224] [-0.90672875  0.42171432]
iteration=2 | best_lap_time=11.13s | best_gains=array([0.00155839, 0.00582564])
[0.05110187 0.32346296] [0.15604828 0.98774943]
iteration=3 | best_lap_time=11.13s | best_gains=array([0.00150729, 0.00550218])
[0.01877029 0.27540051] [0.06799857 0.99768542]
iteration=4 | best_lap_time=11.13s | best_gains=array([0.00148852, 0.00522678])
[ 0.08706308 -0.25127943] [-0.32738502  0.94489102]
iteration=5 | best_lap_time=11.13s | best_gains=array([0.00140145, 0.00547806])
[ 1.10628146 -0.57188318] [-0.8883262   0.45921298]
iteration=6 | best_lap_time=11.13s | best_gains=array([0.00029517, 0.00604994])
[ 9.65281524 -2.41971813] [-0.96998828  0.24315168]
iteration=7 | best_lap_time=11.13s | best_gains=array([0.00935764, 0.00846966])
[ 0.43362985 -1.17767714] [-0.34552911  0.93840803]
iteration=8 | best_lap_time=10.93s

### Going further

- play with different models, and with their hyperparameters
- play with the discount factor
- play with the number of data collected/used
- combine data from random policy with data from trained model

## Conclusion

What we have seen in this notebook:
- collecting data using a random agent in a gym environment
- predicting q-values using a regression model
- the fitted q-iteration (FQI) algorithm to learn from an offline dataset